# HR Policies RAG — Hybrid Retrieval + Re-ranking Assistant

**Week 2 Assignment · Retrieval-Augmented Generation over enterprise HR documents.**

A single-notebook HR assistant that uses **hybrid retrieval (keyword + semantic) with
cross-encoder re-ranking**, then answers with cited sources — and is stress-tested on
15 questions (direct, ambiguous, multi-document, unanswerable).

**Pipeline:**
load `hr_knowledge_base.json` → chunk → embed (**Nebius `Qwen/Qwen3-Embedding-8B`, 4096-dim**) → index in Pinecone (cosine)
→ **hybrid retrieve** (BM25 keyword + dense vector, fused) → **re-rank** (FlashRank cross-encoder) → top-4
→ generate with Claude `claude-opus-4-8` (answer-from-context-only + citations).

> Embeddings run on **Nebius Token Factory** (assignment requirement); generation on **Claude**.
> Hybrid retrieval + re-ranking are the two-stage retrieval upgrade over plain vector search.


## 1. Setup

Load environment variables and confirm the three required keys are present (and not left as placeholders).

In [1]:
import os
import json
import textwrap

from dotenv import load_dotenv

load_dotenv()  # reads .env (copy from .env.demo and fill in your keys)

# Nebius -> embeddings, Anthropic -> generation, Pinecone -> vector store.
# Catch BOTH missing keys and keys left as the .env.demo placeholder ("...").
PLACEHOLDERS = {"", "...", "sk-...", "sk-ant-..."}
for key in ("NEBIUS_API_KEY", "ANTHROPIC_API_KEY", "PINECONE_API_KEY"):
    val = os.environ.get(key, "")
    assert val and val not in PLACEHOLDERS, (
        f"{key} is missing or still the placeholder value. Edit your .env file, "
        f"paste your real key, then restart the kernel and re-run this cell."
    )

INDEX_NAME = os.environ.get("PINECONE_INDEX_NAME", "hr-policies-rag")
EMBED_MODEL = "Qwen/Qwen3-Embedding-8B"   # Nebius Token Factory embedding model
EMBED_DIM = 4096                          # native dimension of Qwen3-Embedding-8B; must match the Pinecone index
GEN_MODEL = "claude-opus-4-8"
RERANK_MODEL = "ms-marco-MiniLM-L-12-v2"  # FlashRank cross-encoder (downloads ~22MB on first run)

print("Environment OK. Index:", INDEX_NAME)


Environment OK. Index: hr-policies-rag


## 2. Load the knowledge base

Read the HR documents into LangChain `Document` objects, carrying their metadata (so we can cite sources later).

In [2]:
from langchain_core.documents import Document

with open("hr_knowledge_base.json") as f:
    kb = json.load(f)

docs = [
    Document(
        page_content=d["content"],
        metadata={**d["metadata"], "id": d["id"]},
    )
    for d in kb["documents"]
]

print(f"Loaded {len(docs)} documents from '{kb['company']}' knowledge base.")
print("Source documents:", sorted({d.metadata["source_document"] for d in docs}))
docs[0]


Loaded 29 documents from 'Northwind Labs' knowledge base.
Source documents: ['Benefits & Compensation', 'Code of Conduct', 'Leave & PTO Policy', 'Onboarding Guide', 'Remote & Hybrid Work Policy']


Document(metadata={'doc_type': 'policy', 'policy_area': 'leave', 'title': 'PTO Accrual', 'source_document': 'Leave & PTO Policy', 'id': 'leave-01'}, page_content='Full-time employees at Northwind Labs accrue 20 days (160 hours) of paid time off (PTO) per calendar year. PTO accrues monthly at a rate of 1.67 days per completed month of service and is available for use as it accrues. Employees hired mid-year accrue PTO on a prorated basis from their start date. Part-time employees accrue PTO proportionally to their scheduled hours. PTO covers vacation and personal days; sick leave is tracked separately under the Sick Leave policy.')

## 3. Chunk

Split documents into overlapping chunks. **500 characters, 100 overlap.** The same `chunks` feed both the dense (Pinecone) index and the in-memory BM25 keyword index.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)

print(f"{len(docs)} documents -> {len(chunks)} chunks")
print("Example chunk:\n", chunks[0].page_content[:300])


29 documents -> 30 chunks
Example chunk:
 Full-time employees at Northwind Labs accrue 20 days (160 hours) of paid time off (PTO) per calendar year. PTO accrues monthly at a rate of 1.67 days per completed month of service and is available for use as it accrues. Employees hired mid-year accrue PTO on a prorated basis from their start date. 


## 4. Embed + index into Pinecone

Embed each chunk with **Nebius `Qwen/Qwen3-Embedding-8B`** (4096-dim) and upsert into a Pinecone serverless index (cosine). Created automatically if absent, and we **wait until it is ready** before upserting.

> Embeddings go through **Nebius Token Factory** (the required Nebius call) via an inline batched-embeddings class (`NebiusBatchEmbeddings`, defined in the next cell) — one batched request instead of one-per-chunk (~100x faster). Deterministic IDs mean re-running **overwrites** rather than duplicates.

In [4]:
import time

from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

from langchain_core.embeddings import Embeddings
from openai import OpenAI


# --- Nebius Token Factory embeddings (batched) --------------------------------
# Sends all chunk texts in ONE request to Nebius's OpenAI-compatible endpoint.
# langchain-nebius sent one request per chunk (rate-limited -> ~12 min for 30 chunks);
# this batched version takes ~6s. Still a genuine Nebius call (same base URL,
# NEBIUS_API_KEY, and Qwen3-Embedding-8B model).
class NebiusBatchEmbeddings(Embeddings):
    def __init__(self, model=EMBED_MODEL, api_key=None,
                 base_url="https://api.tokenfactory.nebius.com/v1/", batch_size=100):
        self.model = model
        self.batch_size = batch_size
        self.client = OpenAI(base_url=base_url, api_key=api_key or os.environ["NEBIUS_API_KEY"])

    def embed_documents(self, texts):
        vectors = []
        for start in range(0, len(texts), self.batch_size):
            resp = self.client.embeddings.create(model=self.model, input=texts[start:start + self.batch_size])
            vectors.extend(item.embedding for item in resp.data)
        return vectors

    def embed_query(self, text):
        return self.client.embeddings.create(model=self.model, input=[text]).data[0].embedding

embeddings = NebiusBatchEmbeddings(model=EMBED_MODEL)

# Quick check that the Nebius embedding call works and returns the expected dimension.
probe = embeddings.embed_query("How much PTO do I get?")
print(f"Nebius embedding OK -- vector dimension = {len(probe)} (expected {EMBED_DIM})")
assert len(probe) == EMBED_DIM, "Embedding dim != EMBED_DIM; update EMBED_DIM to match the model."

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
existing = [idx["name"] for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    print(f"Creating index '{INDEX_NAME}' ({EMBED_DIM}-dim, cosine)...")
    pc.create_index(
        name=INDEX_NAME, dimension=EMBED_DIM, metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
else:
    print(f"Index '{INDEX_NAME}' already exists -- reusing it.")

# Wait until the index is ready before upserting (a fresh serverless index is not
# immediately writable -- upserting too early can hang).
while not pc.describe_index(INDEX_NAME).status["ready"]:
    print("  waiting for index to become ready...")
    time.sleep(2)

ids = [f"{c.metadata['id']}-chunk-{i}" for i, c in enumerate(chunks)]
t0 = time.time()
vectorstore = PineconeVectorStore.from_documents(
    chunks, embedding=embeddings, index_name=INDEX_NAME, ids=ids,
)
print(f"Upserted {len(chunks)} chunks in {time.time() - t0:.1f}s.")


/Users/Mamta/Documents/Mastering Agentic AI /Week 2 Assignment /RAG Implementation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Nebius embedding OK -- vector dimension = 4096 (expected 4096)
Index 'hr-policies-rag' already exists -- reusing it.


Upserted 30 chunks in 6.3s.


In [5]:
# Confirm the vectors landed in the index (count may lag a few seconds after upsert).
index = pc.Index(INDEX_NAME)
time.sleep(5)
index.describe_index_stats()


{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 30}},
 'total_vector_count': 30,
 'vector_type': 'dense'}

## 5. Hybrid retrieval + re-ranking

This is the two-stage retrieval upgrade over plain vector search:

1. **Hybrid retrieve** — combine two complementary retrievers over the same chunks:
   - **Dense / semantic** (Pinecone + Nebius embeddings) — matches *meaning*.
   - **Sparse / keyword** (BM25) — matches *exact terms* like "401(k)" or a phone number.
   - Merged with LangChain's `EnsembleRetriever` (Reciprocal Rank Fusion) into one candidate pool.
2. **Re-rank** — a **FlashRank cross-encoder** scores each candidate against the query and
   keeps the best `top_n`. A cross-encoder reads the query and passage *together*, so it
   judges relevance far more precisely than the first-stage similarity scores.

Net effect: cast a wide net (hybrid), then keep only the most relevant few (re-rank).

In [6]:
import re

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from flashrank import Ranker, RerankRequest

# --- Stage 1: hybrid retrieval -------------------------------------------------
# Dense (semantic) retriever over Pinecone.
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

# Sparse (keyword) retriever -- BM25 over the same chunks, in memory.
# Custom tokenizer (lowercase + split on word characters) so terms like "401(k)"
# match regardless of punctuation. The default tokenizer keeps "401(k)?" as one
# token and fails to match the "401(k)" in the documents.
def _bm25_tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=_bm25_tokenize)
bm25_retriever.k = 8

# Fuse them with Reciprocal Rank Fusion. Dense is weighted a bit higher; BM25 is the
# keyword booster that rescues exact-term matches the embeddings rank too low.
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6],
)

# --- Stage 2: cross-encoder re-ranker -----------------------------------------
reranker = Ranker(model_name=RERANK_MODEL)  # downloads ~22MB the first time


def retrieve(query, fetch_k=10, top_n=4):
    """Hybrid-retrieve a candidate pool, then cross-encoder re-rank to the best top_n."""
    candidates = hybrid_retriever.invoke(query)[:fetch_k]
    passages = [{"id": i, "text": d.page_content} for i, d in enumerate(candidates)]
    ranked = reranker.rerank(RerankRequest(query=query, passages=passages))
    return [candidates[r["id"]] for r in ranked[:top_n]]


print("Hybrid retriever + FlashRank re-ranker ready.")


/var/folders/_r/0466mjws7xgbc35px95_0tx80000gp/T/ipykernel_64783/1019389801.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Hybrid retriever + FlashRank re-ranker ready.


### 5b. See the difference — dense-only vs. hybrid + re-rank

Question 11 of the stress test needs the **401(k)** chunk, which plain vector search ranks just outside the top-4. Watch hybrid + re-ranking pull it in.

In [7]:
demo_q = "If I take parental leave, what happens to my health insurance and 401(k)?"

print("DENSE-ONLY top-4:")
for d in dense_retriever.invoke(demo_q)[:4]:
    print(f"  - {d.metadata['title']}  ({d.metadata['source_document']})")

print("\nHYBRID + RE-RANK top-4:")
for d in retrieve(demo_q):
    print(f"  - {d.metadata['title']}  ({d.metadata['source_document']})")


DENSE-ONLY top-4:


INFO:httpx:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


  - Parental Leave  (Leave & PTO Policy)
  - Parental Leave  (Leave & PTO Policy)
  - Life and Disability Insurance  (Benefits & Compensation)
  - Sick Leave  (Leave & PTO Policy)

HYBRID + RE-RANK top-4:


INFO:httpx:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


  - Parental Leave  (Leave & PTO Policy)
  - Parental Leave  (Leave & PTO Policy)
  - Bereavement Leave  (Leave & PTO Policy)
  - 401(k) Retirement Plan  (Benefits & Compensation)


## 6. Generation chain (Claude)

The prompt forces Claude to answer **only from the retrieved context** and to explicitly decline when the answer isn't there — this is what makes the *unanswerable* questions fail gracefully instead of hallucinating. `ask()` runs the full hybrid + re-rank retrieval, then generates, and returns the answer **and** the re-ranked source chunks (citations).

In [8]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

llm = ChatAnthropic(model=GEN_MODEL, max_tokens=1024)

SYSTEM = (
    "You are an HR policy assistant for Northwind Labs. Answer the employee's "
    "question using ONLY the context below, retrieved from the official HR "
    "knowledge base.\n\n"
    "Rules:\n"
    "- Use only facts present in the context. Do not use outside knowledge or guess.\n"
    "- If the context does not contain the answer, reply EXACTLY: \"I'm sorry, that "
    "topic isn't covered in the HR knowledge base. Please contact HR for help.\" "
    "Do not invent a policy.\n"
    "- If the question is ambiguous, briefly note the interpretations the context supports.\n"
    "- Be concise and name the policy titles you used.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM), ("human", "{question}")])


def format_docs(retrieved):
    return "\n\n".join(
        f"[{d.metadata.get('title', '?')} -- {d.metadata.get('source_document', '?')}]\n{d.page_content}"
        for d in retrieved
    )


def ask(question, top_n=4):
    """Hybrid-retrieve + re-rank, generate, and return (answer, reranked_docs)."""
    retrieved = retrieve(question, top_n=top_n)
    messages = prompt.format_messages(context=format_docs(retrieved), question=question)
    answer = llm.invoke(messages).content
    return answer, retrieved


# Sanity check
answer, sources = ask("How much PTO do I get each year?")
print(answer)
print("\nSources:", [d.metadata["title"] for d in sources])


INFO:httpx:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Full-time employees at Northwind Labs accrue **20 days (160 hours) of PTO per calendar year**, accruing monthly at 1.67 days per completed month of service.

A few notes:
- If you're hired mid-year, PTO is prorated from your start date.
- Part-time employees accrue PTO proportionally to their scheduled hours.

(Source: PTO Accrual — Leave & PTO Policy)

Sources: ['PTO Accrual', 'PTO Carryover', 'Sick Leave', 'Wellness Reimbursement']


## 7. The 15-question stress test

Four buckets:
- **Direct factual** (6) — single-document, retrieval should succeed.
- **Ambiguous** (3) — underspecified wording.
- **Multi-document** (3) — answer requires combining two source documents.
- **Unanswerable** (3) — topics deliberately omitted from the KB; success = the system declines.


In [9]:
questions = [
    # --- Direct factual ---
    {"category": "Direct factual", "q": "How many PTO days do I accrue per year?"},
    {"category": "Direct factual", "q": "What is the company 401(k) match?"},
    {"category": "Direct factual", "q": "How many weeks of paid parental leave does the primary caregiver get?"},
    {"category": "Direct factual", "q": "How often are employees paid?"},
    {"category": "Direct factual", "q": "What is the home office equipment stipend for remote employees?"},
    {"category": "Direct factual", "q": "How do I report harassment?"},
    # --- Ambiguous ---
    {"category": "Ambiguous", "q": "What's the policy on time off?"},
    {"category": "Ambiguous", "q": "Can I work from home?"},
    {"category": "Ambiguous", "q": "What's the deadline?"},
    # --- Multi-document ---
    {"category": "Multi-document", "q": "I'm a new remote hire -- what do I need to set up and what are the rules for working remotely in my first weeks?"},
    {"category": "Multi-document", "q": "If I take parental leave, what happens to my health insurance and 401(k)?"},
    {"category": "Multi-document", "q": "What can I get reimbursed for as a remote employee?"},
    # --- Unanswerable (not in the KB) ---
    {"category": "Unanswerable", "q": "What is the relocation reimbursement policy?"},
    {"category": "Unanswerable", "q": "Do we offer pet insurance?"},
    {"category": "Unanswerable", "q": "What is the sabbatical policy after 5 years?"},
]
len(questions)


15

In [10]:
for i, item in enumerate(questions, 1):
    answer, sources = ask(item["q"])
    print("=" * 100)
    print(f"Q{i} [{item['category']}]: {item['q']}")
    print("-" * 100)
    print("ANSWER:")
    print(textwrap.fill(answer, 100))
    print("\nRE-RANKED SOURCES (top 4):")
    for d in sources:
        print(f"  - {d.metadata.get('title')}  ({d.metadata.get('source_document')})  [id={d.metadata.get('id')}]")
    print()


INFO:httpx:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Q1 [Direct factual]: How many PTO days do I accrue per year?
----------------------------------------------------------------------------------------------------
ANSWER:
Full-time employees at Northwind Labs accrue **20 days (160 hours)** of PTO per calendar year,
accruing monthly at 1.67 days per completed month of service. If you were hired mid-year, your PTO
is prorated from your start date, and part-time employees accrue proportionally to their scheduled
hours.  (Source: PTO Accrual — Leave & PTO Policy)

RE-RANKED SOURCES (top 4):
  - PTO Accrual  (Leave & PTO Policy)  [id=leave-01]
  - PTO Carryover  (Leave & PTO Policy)  [id=leave-02]
  - Sick Leave  (Leave & PTO Policy)  [id=leave-03]
  - Bereavement Leave  (Leave & PTO Policy)  [id=leave-05]



INFO:httpx:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Q14 [Unanswerable]: Do we offer pet insurance?
----------------------------------------------------------------------------------------------------
ANSWER:
I'm sorry, that topic isn't covered in the HR knowledge base. Please contact HR for help.

RE-RANKED SOURCES (top 4):
  - Health Insurance  (Benefits & Compensation)  [id=benefits-01]
  - Dental and Vision  (Benefits & Compensation)  [id=benefits-02]
  - Life and Disability Insurance  (Benefits & Compensation)  [id=benefits-04]
  - Parental Leave  (Leave & PTO Policy)  [id=leave-04]



INFO:httpx:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Q15 [Unanswerable]: What is the sabbatical policy after 5 years?
----------------------------------------------------------------------------------------------------
ANSWER:
I'm sorry, that topic isn't covered in the HR knowledge base. Please contact HR for help.

RE-RANKED SOURCES (top 4):
  - Bereavement Leave  (Leave & PTO Policy)  [id=leave-05]
  - PTO Carryover  (Leave & PTO Policy)  [id=leave-02]
  - Parental Leave  (Leave & PTO Policy)  [id=leave-04]
  - Sick Leave  (Leave & PTO Policy)  [id=leave-03]



## 8. Findings

Record the outcome of each question (success / partial / fail) and the reason in
[`evaluation.md`](./evaluation.md), using the printed answers and re-ranked sources above.

**What to look for:**
- *Direct factual* → correct, sourced answers.
- *Ambiguous* → does it surface multiple interpretations, or silently pick one?
- *Multi-document* → did hybrid + re-ranking retrieve chunks from **both** needed documents? (compare with §5b).
- *Unanswerable* → did it return the "not covered" decline instead of fabricating a policy? (the key correctness check)

**This notebook already implements** hybrid retrieval (BM25 + dense) and cross-encoder
re-ranking. Further tuning levers: the fusion `weights`, `fetch_k` / `top_n`, the
re-ranker model, and metadata filtering by `policy_area`.
